# Calibração NECr × S: o que os partidos fizeram, não só se acertaram a expectativa

## Enquadramento

As análises do Cap. 3 até aqui avaliam a coordenação intrapartidária contra uma
**expectativa ex-ante**: NECr/Mp, NECr/(Mp+1) e a TAA usam a bancada estadual prévia ($M_p$)
como referência do "quantos candidatos viáveis o partido deveria ter". Isso é informativo,
mas carrega um risco apontado pelo orientador na reunião de 28/08/2026 (ver
`notes/anotacoes.md`):

> Ok. Mas o ponto é avaliar o que os partidos fazem e não só se acertam na avaliação prévia
> sobre quantos candidatos têm de fato chance. Imagine que NEC seja > 1, digamos igual a 3,
> e o partido de fato eleja 3 pessoas. Mesmo que a expectativa fosse outra qualquer, ele
> gastou para 3 e elegeu 3. Isso em si já é um dado legal, não?

Este notebook opera essa provocação: em vez de comparar a alocação de recursos com $M_p$,
compara-a com o **resultado realizado** ($S$ = cadeiras conquistadas pela lista). Duas
perguntas, deliberadamente mantidas separadas:

1. **Calibração de escala** — o partido gastou para *quantos* candidatos viáveis (NECr)
   próximo de *quantos* efetivamente elegeu ($S$)? ($\text{gap} = \text{NECr} - S$;
   $\text{razão} = \text{NECr}/S$)
2. **Discriminação** — dentre os $S$ candidatos mais financiados, quantos estavam entre os
   eleitos? (`taa_expost_S`, de `cap3_taa_features.py` — lá é teste de robustez; aqui é
   resultado principal)

**Ressalva antitautológica (Bolognesi et al. 2020; Guarnieri & Silva 2025):** $S$ entra
como **alvo/desfecho avaliado**, nunca como preditor de um modelo causal. Isso é material do
**Cap. 3** (validação/caracterização da medida de coordenação) — nada aqui deve migrar para
os modelos de montante (regressão fracionária) ou timing (sobrevivência) dos caps. 4-5, que
seguem proibidos de usar `eleito` no lado direito das equações.

**Ressalva de endogeneidade:** dinheiro pode causar voto, então a proximidade NECr↔$S$
mistura "o partido previu bem" com "o repasse ajudou a eleger". A leitura do orientador
contorna isso — é um fato conjunto sobre comportamento e desfecho, não uma alegação causal —
mas o notebook não deve ser lido como prova de acerto preditivo em sentido forte.

Toda a engenharia de features está em `src/2_gold/cap3_calibracao_features.py`, que
reaproveita `_preparar` e `acertos_fracionarios` de `cap3_taa_features.py` e
`necr_cortes`/`necr_excedente` de `cap3_necr_decomposicao.py`.

In [1]:
import os
from pathlib import Path
ROOT = Path().resolve().parent  # notebooks/ -> project root
os.chdir(ROOT)

In [2]:
import sys
import warnings

import numpy as np
import pandas as pd
import statsmodels.api as sm
import plotly.express as px

sys.path.insert(0, str(Path('src/2_gold').resolve()))
from cap3_calibracao_features import (
    construir_calibracao,
    resumo_calibracao,
    calibrado,
    discrimina_bem,
    classificar_tipologia,
    decompor_variancia_gap,
)
from cap3_cs_features import carregar_rrd

warnings.filterwarnings('ignore')
pd.set_option('display.width', 140)

painel = construir_calibracao(carregar_rrd())
print(f"{len(painel):,} listas (partido x UF x ano) com algum recurso partidário distribuído")
painel.groupby('ano_eleicao').size().to_frame('n_listas')

1,942 listas (partido x UF x ano) com algum recurso partidário distribuído


,n_listas
ano_eleicao,
2014,508
2018,786
2022,648


## 1. Cobertura do painel

Universo: listas (partido × UF × ano) com `vr_receita_recursos_partidos` total > 0.
Diferente de `construir_taa()`, **não** se exige $M_p \geq 1$ — aqui $M_p$ é só referência
comparativa, não denominador do indicador; um partido sem bancada estadual prévia também faz
uma aposta ao distribuir recursos de forma desigual entre candidatos, e é essa aposta que a
calibração avalia.

In [3]:
cobertura = painel.groupby('ano_eleicao').agg(
    n_listas=('S', 'size'),
    pct_sem_eleitos=('sem_eleitos', 'mean'),
    S_mediana=('S', 'median'),
    NECr_mediana=('NECr', 'median'),
    n_com_recursos_mediana=('n_com_recursos', 'median'),
).round(3)
print("Cobertura do painel por ano — note que a maioria das listas não elege ninguém "
      "(50%->68%), o que motiva tratar S=0 como ramo próprio (Seção 4), não como exceção.")
cobertura

Cobertura do painel por ano — note que a maioria das listas não elege ninguém (50%->68%), o que motiva tratar S=0 como ramo próprio (Seção 4), não como exceção.


,n_listas,pct_sem_eleitos,S_mediana,NECr_mediana,n_com_recursos_mediana
ano_eleicao,,,,,
2014,508,0.500,0.5,1.081,2.0
2018,786,0.618,0.0,1.878,3.0
2022,648,0.681,0.0,4.814,9.0


## 2. Calibração de escala: NECr ≈ S?

Comparam-se o gap ($\text{NECr}-S$), a razão ($\text{NECr}/S$) e a % de listas "calibradas"
(razão dentro de $[2/3,\ 3/2]$, i.e. o partido gastou como se fosse eleger entre 67% e 150%
do que de fato elegeu) — restrito a $S \geq 1$.

Repete-se o exercício para as variantes de `cap3_necr_decomposicao.py` (limiares de
relevância `NECr_0…NECr_05` e `NECr_excedente`, que descontam o piso distributivo do FEFC),
porque o NECr bruto pode subir só por margem extensiva: em 2022, 88,8% dos candidatos
recebem *algum* recurso partidário (37,9% em 2014), o que infla o NECr bruto sem que a
concentração dos recursos eleitoralmente decisivos tenha mudado na mesma proporção.

In [4]:
variantes = ['NECr', 'NECr_0', 'NECr_005', 'NECr_01', 'NECr_02', 'NECr_05', 'NECr_excedente']

resumos = {v: resumo_calibracao(painel, variante=v).set_index('ano_eleicao') for v in variantes}

print("Correlação NECr x S, por variante e ano:")
display(pd.DataFrame({v: r['corr_NECr_S'] for v, r in resumos.items()}).round(3))

print("\n% de listas calibradas (razão dentro de [2/3, 3/2]), por variante e ano:")
display(pd.DataFrame({v: r['pct_calibrado'] for v, r in resumos.items()}).round(3))

print("\nGap mediano (NECr - S), por variante e ano:")
pd.DataFrame({v: r['gap_mediano'] for v, r in resumos.items()}).round(3)

Correlação NECr x S, por variante e ano:


,NECr,NECr_0,NECr_005,NECr_01,NECr_02,NECr_05,NECr_excedente
ano_eleicao,,,,,,,
2014,0.293,0.293,0.293,0.293,0.293,0.151,0.555
2018,0.674,0.674,0.674,0.673,0.675,0.662,0.670
2022,0.560,0.560,0.560,0.559,0.564,0.376,0.592



% de listas calibradas (razão dentro de [2/3, 3/2]), por variante e ano:


,NECr,NECr_0,NECr_005,NECr_01,NECr_02,NECr_05,NECr_excedente
ano_eleicao,,,,,,,
2014,0.642,0.642,0.642,0.642,0.642,0.646,0.350
2018,0.423,0.423,0.423,0.423,0.423,0.423,0.360
2022,0.106,0.106,0.106,0.106,0.101,0.097,0.198



Gap mediano (NECr - S), por variante e ano:


,NECr,NECr_0,NECr_005,NECr_01,NECr_02,NECr_05,NECr_excedente
ano_eleicao,,,,,,,
2014,0.000,0.000,0.000,0.000,0.000,0.000,0.000
2018,0.908,0.908,0.908,0.908,0.908,0.919,0.867
2022,4.402,4.402,4.402,4.402,4.402,4.402,2.996


**Leitura:** mesmo descontando margem extensiva, a calibração de escala se deteriora
2014→2022 — o gap mediano do NECr do excedente ainda salta de 0 para ~3 cadeiras, e a %
calibrada cai de 35% para 20%. A correlação NECr×S, por outro lado, **sobe** de 0,29 para
0,56–0,67 em todas as variantes: o partido erra cada vez mais o *nível* (gasta para muito
mais candidatos "viáveis" do que efetivamente elege), mas ordena cada vez melhor *quem* entre
eles vai se eleger — a mesma dissociação nível/ordenação que sustenta o achado central da
TAA (`3_taa_alocacao.ipynb`, `taa` vs. `taa_outros`).

## 3. Teste formal de calibração: $S = \alpha + \beta\,\text{NECr}$

Calibração perfeita implicaria $\alpha=0,\ \beta=1$. Regride-se $S$ (contagem observada)
sobre o NECr bruto, por ano, com erros HC3, e testa-se a hipótese conjunta
$H_0:\alpha=0 \wedge \beta=1$ (teste de Wald).

In [5]:
linhas = []
for ano in [2014, 2018, 2022]:
    sub = painel[(painel.ano_eleicao == ano) & (painel.S > 0)]
    X = sm.add_constant(sub['NECr'].values)
    y = sub['S'].values
    m = sm.OLS(y, X).fit(cov_type='HC3')
    wald = m.wald_test((np.eye(2), np.array([0, 1])), scalar=True)
    linhas.append({
        'ano_eleicao': ano, 'N': len(sub),
        'alpha': m.params[0], 'beta': m.params[1], 'R2': m.rsquared,
        'wald_p_(a=0,b=1)': float(wald.pvalue),
    })

tab_calib = pd.DataFrame(linhas)
print("H0: alpha=0, beta=1 (calibração perfeita) é rejeitada em todos os anos (p < 0,001).")
print("beta < 1 em todos os anos: cada NECr adicional se traduz em menos de 1 cadeira a "
      "mais - o excesso de 'candidatos viáveis' financiados não vira cadeira 1-para-1.")
tab_calib.round(3)

H0: alpha=0, beta=1 (calibração perfeita) é rejeitada em todos os anos (p < 0,001).
beta < 1 em todos os anos: cada NECr adicional se traduz em menos de 1 cadeira a mais - o excesso de 'candidatos viáveis' financiados não vira cadeira 1-para-1.


,ano_eleicao,N,alpha,beta,R2,"wald_p_(a=0,b=1)"
0,2014,254,1.536,0.157,0.086,0.0
1,2018,300,0.690,0.298,0.454,0.0
2,2022,207,0.751,0.226,0.313,0.0


## 4. O ramo $S=0$: gastou para vários, não elegeu nenhum

Listas com $S=0$ não são exceção — são a maioria em 2018 e 2022 (61,8% e 68,1%). Não entram
na calibração de escala (razão indefinida), mas são o caso mais informativo sobre
descoordenação: o partido efetivamente distribuiu recursos de forma não-degenerada
(`NECr > 1` é possível mesmo aqui) e não converteu nada em cadeira.

In [6]:
perfil_s0 = (
    painel[painel.sem_eleitos]
    .groupby('ano_eleicao')
    .agg(
        n_listas=('NECr', 'size'),
        NECr_mediano=('NECr', 'median'),
        n_com_recursos_mediano=('n_com_recursos', 'median'),
        vr_total_mediano=('vr_total_lista', 'median'),
    )
    .round(2)
)
print("Perfil das listas S=0, por ano:")
display(perfil_s0)

print("\nCasos extremos 2022 — maior NECr entre listas que não elegeram ninguém "
      "(candidatas a estudo de caso: dinheiro bem distribuído, zero conversão):")
painel[(painel.ano_eleicao == 2022) & painel.sem_eleitos].nlargest(8, 'NECr')[
    ['sg_uf', 'sg_partido', 'NECr', 'n_com_recursos', 'vr_total_lista', 'qt_vaga']
]

Perfil das listas S=0, por ano:


,n_listas,NECr_mediano,n_com_recursos_mediano,vr_total_mediano
ano_eleicao,,,,
2014,254,1.00,1.0,46244.58
2018,486,1.68,2.0,147987.64
2022,441,3.97,8.0,555700.00



Casos extremos 2022 — maior NECr entre listas que não elegeram ninguém (candidatas a estudo de caso: dinheiro bem distribuído, zero conversão):


,sg_uf,sg_partido,NECr,n_com_recursos,vr_total_lista,qt_vaga
1896,SP,PATRIOTA,31.805442,70,10566186.01,70
1727,RJ,DC,30.672087,41,380318.90,46
1905,SP,PRTB,30.099523,57,255025.22,70
1383,BA,PMB,30.024770,31,14027.63,39
1538,MG,NOVO,24.324050,30,2117662.00,53
1830,RS,PSC,20.547842,26,740000.00,31
1706,PR,PMB,18.306099,24,67559.00,30
1822,RS,PATRIOTA,18.029779,22,685000.00,31


## 5. Discriminação: dentre os $S$ mais financiados, quantos se elegeram?

`taa_expost_S` — aqui promovida a resultado principal, não robustez como em
`3_taa_alocacao.ipynb` — mede a fração dos $S$ candidatos mais financiados que de fato se
elegeu. Comparada ao benchmark aleatório $S/n$ (probabilidade de acerto sorteando $S$ nomes
ao acaso entre os $n$ candidatos da lista).

In [7]:
sub = painel[painel.S > 0].copy()
sub['benchmark_aleatorio'] = sub['S'] / sub['n_cands']

disc = sub.groupby('ano_eleicao')[['taa_expost_S', 'benchmark_aleatorio']].mean().round(3)
disc['vantagem_sobre_acaso'] = (disc['taa_expost_S'] - disc['benchmark_aleatorio']).round(3)
print("Discriminação vs. benchmark aleatório, por ano (S >= 1):")
disc

Discriminação vs. benchmark aleatório, por ano (S >= 1):


,taa_expost_S,benchmark_aleatorio,vantagem_sobre_acaso
ano_eleicao,,,
2014,0.730,0.346,0.384
2018,0.764,0.341,0.423
2022,0.666,0.163,0.503


**Leitura:** a discriminação supera o acaso com folga larga nos três ciclos — e a vantagem
cresce (0,38 em 2014 → 0,50 em 2022), não é constante — mas já em 2014, pré-FEFC, dinheiro de qualquer origem
identificava razoavelmente bem quem ia se eleger antes do fundo existir. O achado do FEFC
está na Seção 2/3 (calibração de escala) e em `3_taa_alocacao.ipynb` (`taa` supera
`taa_outros` só a partir de 2018): o fundo não criou a capacidade de discriminar quem vai
ganhar, ele mudou a relação entre volume distribuído e volume necessário.

## 6. Mapa 2D: calibração × discriminação

Cruzam-se as duas dimensões numa tipologia $2\times2$ (só definida para $S\geq1$; ver
Seção 4 para $S=0$):

| | discrimina bem ($\text{taa\_expost\_S}\geq0{,}5$) | discrimina mal |
|---|---|---|
| **calibrado** (razão $\in[2/3,\,3/2]$) | Coordenação eficaz | Acertou o alvo, descalibrou a escala |
| **descalibrado** | Acertou a escala, errou o alvo | Descoordenado |

In [8]:
tipo = classificar_tipologia(painel)
tab_tipo = pd.crosstab(painel['ano_eleicao'], tipo)
ordem_tipo = ['Coordenação eficaz', 'Acertou o alvo, descalibrou a escala',
              'Acertou a escala, errou o alvo', 'Descoordenado']
tab_tipo = tab_tipo[[c for c in ordem_tipo if c in tab_tipo.columns]]
print("Contagens por tipologia e ano (S >= 1):")
display(tab_tipo)
print("\n% por ano:")
tab_tipo.div(tab_tipo.sum(axis=1), axis=0).round(3)

Contagens por tipologia e ano (S >= 1):


tipologia,Coordenação eficaz,"Acertou o alvo, descalibrou a escala","Acertou a escala, errou o alvo",Descoordenado
ano_eleicao,,,,
2014,139,61,24,30
2018,122,124,5,49
2022,20,141,2,44



% por ano:


tipologia,Coordenação eficaz,"Acertou o alvo, descalibrou a escala","Acertou a escala, errou o alvo",Descoordenado
ano_eleicao,,,,
2014,0.547,0.240,0.094,0.118
2018,0.407,0.413,0.017,0.163
2022,0.097,0.681,0.010,0.213


In [9]:
plot_df = painel[painel.S > 0].copy()
plot_df['tipologia'] = tipo[plot_df.index]
plot_df['ano_eleicao'] = plot_df['ano_eleicao'].astype(str)
plot_df['razao_NECr_clip'] = plot_df['razao_NECr'].clip(upper=6)

fig = px.scatter(
    plot_df, x='razao_NECr_clip', y='taa_expost_S', color='tipologia',
    facet_col='ano_eleicao', hover_data=['sg_uf', 'sg_partido', 'S', 'NECr'],
    labels={'razao_NECr_clip': 'razão NECr / S (truncada em 6)',
            'taa_expost_S': 'discriminação (taa_expost_S)'},
    title='Calibração x discriminação, por lista e ano',
    category_orders={'tipologia': ordem_tipo},
)
fig.add_vline(x=2/3, line_dash='dot', line_color='gray')
fig.add_vline(x=3/2, line_dash='dot', line_color='gray')
fig.add_hline(y=0.5, line_dash='dot', line_color='gray')
fig.update_layout(height=430)
fig.show()

**Leitura:** a "coordenação eficaz" domina em 2014 (139 listas) e ainda compete em 2018
(122), mas praticamente desaparece em 2022 (20) — substituída por "acertou o alvo,
descalibrou a escala" (141 listas): o partido segue identificando quem vai se eleger, mas
passa a espalhar recursos para muito mais candidatos "viáveis" do que consegue eleger de
fato. É a assinatura que a Seção 2 já indicava em correlações agregadas, agora visível
lista a lista.

## 7. Diferenças entre partidos e UFs

Três perguntas: (a) quais partidos calibram melhor, controlando a composição de anos; (b)
o desalinhamento é atributo de partido (nacional) ou de diretório estadual (UF); (c) o
quanto um mesmo partido varia entre UFs.

**Cuidado:** partidos ativos majoritariamente em 2022 (p.ex. NOVO, UNIÃO, PODE) têm razão
mediana mais alta só porque **todo mundo** descalibra mais em 2022 (Seção 2) — comparar a
razão bruta entre partidos confunde efeito de partido com composição de anos. Por isso o
gap é padronizado (z-score) **dentro de cada ano** antes de ranquear partidos entre si.

In [10]:
sub = painel[painel.S > 0].copy()
sub['gap_z'] = sub.groupby('ano_eleicao')['gap_NECr'].transform(lambda x: (x - x.mean()) / x.std())

ranking = (
    sub.groupby('sg_partido_norm')
    .agg(n_listas=('gap_z', 'size'), gap_z_medio=('gap_z', 'mean'),
         discriminacao_media=('taa_expost_S', 'mean'))
    .query('n_listas >= 5')
    .sort_values('gap_z_medio')
    .round(3)
)
print("Ranking de partidos por gap de calibração padronizado dentro do ano "
      "(mais negativo = concentra menos que o esperado / mais próximo de S; "
      "mais positivo = espalha mais recursos do que converte em cadeiras).")
print("n >= 5 listas em toda a série.\n")
ranking

Ranking de partidos por gap de calibração padronizado dentro do ano (mais negativo = concentra menos que o esperado / mais próximo de S; mais positivo = espalha mais recursos do que converte em cadeiras).
n >= 5 listas em toda a série.



,n_listas,gap_z_medio,discriminacao_media
sg_partido_norm,,,
PL,23,-0.607,0.653
PC DO B,21,-0.450,0.881
PRP,6,-0.385,0.500
PSL,17,-0.258,0.528
PTB,20,-0.251,0.768
PSC,20,-0.223,0.755
DEM,30,-0.201,0.787
PMDB,27,-0.175,0.714
PR,40,-0.152,0.880


**Leitura:** PL, PC do B e PRP calibram melhor que a média (dado o ano); NOVO, PSOL e
UNIÃO descalibram mais — consistente com a estratégia conhecida do NOVO de lançar muitos
candidatos com recursos pulverizados (`discriminacao_media` = 0,125, a mais baixa da
tabela: quase nenhum dos mais financiados pelo NOVO se elege). É hipótese de partido a
examinar qualitativamente, não uma alegação causal desta tabela.

In [11]:
var_gap = decompor_variancia_gap(painel)
print("Decomposição aproximada de variância do gap de calibração (R² por agrupamento), "
      "por ano — responde se o desalinhamento é atributo do partido (nacional), do "
      "diretório estadual (UF), ou de ambos:")
var_gap.round(3)

Decomposição aproximada de variância do gap de calibração (R² por agrupamento), por ano — responde se o desalinhamento é atributo do partido (nacional), do diretório estadual (UF), ou de ambos:


,ano_eleicao,n_listas,R2_partido,R2_uf,R2_partido_uf
0,2014,508,0.138,0.041,0.171
1,2018,786,0.242,0.139,0.377
2,2022,648,0.243,0.172,0.412


**Leitura:** $R^2_{partido} > R^2_{UF}$ nos três anos, e a soma conjunta explica no
máximo ~41% (2022) da variância do gap — a maior parte do desalinhamento é idiossincrática
à lista específica (partido×UF×ano), não um traço estável do partido nacional nem do
diretório estadual isoladamente. Isso pesa contra uma leitura "é sempre esse partido que
descoordena" e a favor de explicações situacionais (força do adversário na UF, disputa
interna daquele ciclo).

In [12]:
disp_intra = (
    painel[painel.S > 0]
    .groupby('sg_partido_norm')['gap_NECr']
    .agg(n_listas='size', dp_gap='std', gap_medio='mean')
    .query('n_listas >= 5')
    .sort_values('dp_gap')
    .round(2)
)
print("Dispersão do gap de calibração entre UFs, por partido (n >= 5 listas na série; "
      "menor dp = partido mais uniforme entre estados):")
disp_intra

Dispersão do gap de calibração entre UFs, por partido (n >= 5 listas na série; menor dp = partido mais uniforme entre estados):


,n_listas,dp_gap,gap_medio
sg_partido_norm,,,
PRP,6,0.34,0.28
PC DO B,21,0.45,0.42
PMN,5,0.96,0.99
DEM,30,1.25,0.57
PPS,13,1.50,0.84
PMDB,27,1.51,-0.06
PTB,20,1.60,0.50
PHS,9,1.64,1.26
PR,40,1.65,0.71


## 8. Sensibilidade

A tolerância de calibração ($\pm50\%$ na razão) e o limiar de discriminação ($\geq50\%$)
são escolhas de desenho. Testam-se alternativas mais e menos exigentes.

In [13]:
print("% calibrado sob tolerâncias alternativas (razão NECr/S dentro do intervalo), por ano:")
sub = painel[painel.S > 0].copy()
for lo, hi, label in [(0.8, 1.25, 'pm25pct'), (2/3, 3/2, 'pm50pct_adotada'), (0.5, 2.0, 'pm100pct')]:
    sub[f'cal_{label}'] = calibrado(sub['razao_NECr'], (lo, hi))

display(
    sub.groupby('ano_eleicao')[[c for c in sub.columns if c.startswith('cal_')]]
    .mean().round(3)
)

print("\n% 'discrimina bem' sob limiares alternativos, por ano:")
for lim in [0.3, 0.5, 0.7]:
    sub[f'disc_{lim}'] = discrimina_bem(sub['taa_expost_S'], lim)
sub.groupby('ano_eleicao')[[c for c in sub.columns if c.startswith('disc_')]].mean().round(3)

% calibrado sob tolerâncias alternativas (razão NECr/S dentro do intervalo), por ano:


,cal_pm25pct,cal_pm50pct_adotada,cal_pm100pct
ano_eleicao,,,
2014,0.539,0.642,0.827
2018,0.310,0.423,0.663
2022,0.058,0.106,0.242



% 'discrimina bem' sob limiares alternativos, por ano:


,disc_0.3,disc_0.5,disc_0.7
ano_eleicao,,,
2014,0.823,0.787,0.650
2018,0.850,0.820,0.693
2022,0.821,0.778,0.546


A ordenação entre anos (2014 mais calibrado que 2022; discriminação estável e acima do
acaso nos três) é robusta às escolhas de tolerância — não é artefato do limiar adotado.

## 9. Síntese e persistência

- A **calibração de escala** piora 2014→2022, mesmo descontando margem extensiva: os
  partidos passaram a financiar de forma não-degenerada muito mais candidatos do que
  conseguem eleger (regressão $S\sim\text{NECr}$ rejeita calibração perfeita nos três anos;
  $\beta<1$ em todos).
- A **discriminação** (os mais financiados são os eleitos) é estável e bate o acaso com
  folga nos três ciclos, inclusive em 2014 — o achado do FEFC não é "criar poder de
  previsão", é mudar a relação escala/resultado (convergente com `3_taa_alocacao.ipynb`).
- A tipologia $2\times2$ lista-a-lista mostra a composição migrando de "coordenação
  eficaz" para "acertou o alvo, descalibrou a escala" ao longo do período.
- O desalinhamento é majoritariamente idiossincrático à lista, não um traço estável do
  partido nacional nem do diretório estadual (R² conjunto ≤ 0,41).
- $S=0$ é o caso modal (50%->68%) e concentra os casos de descoordenação mais extremos.

Painel salvo para reuso em `data/processed/df_calibracao_lista.parquet`
(idêntico ao gerado por `python src/2_gold/cap3_calibracao_features.py`).

In [14]:
destino = ROOT / 'data' / 'processed' / 'df_calibracao_lista.parquet'
painel.to_parquet(destino, index=False)
print(f"salvo: {destino} ({len(painel)} listas, {painel.shape[1]} colunas)")

salvo: D:\recursos-campanha\data\processed\df_calibracao_lista.parquet (1942 listas, 30 colunas)
